In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / "data_collection").is_dir():
    os.chdir(current_dir / "data_collection")
    print("Working directory normalized to data_collection/.")
else:
    print(f"Using working directory: {current_dir}")

# Data Quality & Visual QA — `news_daily_df` + `market_daily_df`

**Academic research only — not investment advice.**

This notebook is the verification counterpart to `01_ravenpack_news_extraction.ipynb` (news gold table) and `02_crsp_sector_etf_price_extraction.ipynb` (market gold table). It does not re-pull from WRDS; it reads the committed CSV outputs and answers three questions:

1. **Is the data clean?** — schema, nulls, duplicate keys, impossible values, internal consistency of derived columns.
2. **Is there enough of it?** — coverage per sector ETF and per session, gaps against the actual trading calendar, news volume stability over time.
3. **Are the timestamps right?** — this is the one that can silently break the whole study. Three separate alignment checks: (a) the 4:00 PM ET after-hours cutoff was applied as specified, (b) every news `session_date` is a real trading session with news dated *before* it, and (c) `fwd_1d_return` / `fwd_5d_return` really are *forward* returns, recomputed from scratch.

Every check ends in a pass/fail row in the scorecard at the bottom. A failure here is a bug in 01/02, not something to work around downstream.

**Licensing note:** the row-level silver table (`raw/ravenpack_core_events_*.csv`) is a licensed WRDS export. This notebook reads *only* its timestamp columns and emits *only* aggregate counts/distributions from them — no headline or article text is loaded or displayed, so no cell output here is restricted data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

pd.options.display.max_columns = 60
pd.options.display.width = 160

NOTEBOOK_DIR = Path.cwd()
RAW_DIR = NOTEBOOK_DIR / "raw"

MARKET_CSV = NOTEBOOK_DIR / "market_daily_df.csv"
NEWS_CSV = NOTEBOOK_DIR / "news_daily_df.csv"
CORE_EVENTS_CSV = RAW_DIR / "ravenpack_core_events_2020_2025.csv"  # silver, gitignored, optional

MARKET_CLOSE_HOUR_ET = 16  # the after-hours cutoff 01_ applied
TOL = 1e-6

# Palette: one blue for magnitude, blue/gray/red as the diverging sentiment triple,
# recessive gray for grid and axes. Colour is never the only channel — every series
# is either directly labelled or the sole series in its panel.
BLUE, RED, GRAY = "#2a78d6", "#e34948", "#898781"
GRID, INK, MUTED = "#e1e0d9", "#0b0b0b", "#52514e"
SEQ = plt.cm.Blues

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": MUTED,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "font.size": 9,
})

# Every check appends here; rendered as a scorecard in the final section.
CHECKS = []


def record(name, passed, detail=""):
    """Record a QA check result and echo it inline."""
    CHECKS.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": str(detail)})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}{(' — ' + str(detail)) if detail else ''}")


market_df = pd.read_csv(MARKET_CSV, parse_dates=["session_date"])
news_df = pd.read_csv(NEWS_CSV, parse_dates=["session_date"])

print(f"market_daily_df: {market_df.shape[0]:,} rows x {market_df.shape[1]} cols  "
      f"({market_df['session_date'].min().date()} to {market_df['session_date'].max().date()})")
print(f"news_daily_df:   {news_df.shape[0]:,} rows x {news_df.shape[1]} cols  "
      f"({news_df['session_date'].min().date()} to {news_df['session_date'].max().date()})")

## 1. Schema, nulls, and key uniqueness

Structural preconditions everything else depends on: `market_daily_df` must be unique on `(session_date, ticker)`, `news_daily_df` must be unique on `session_date`, and neither may carry nulls in the columns that feed the model.

In [ ]:
def profile(df, name):
    prof = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    print(f"--- {name} ---")
    display(prof)
    return prof


market_prof = profile(market_df, "market_daily_df")
news_prof = profile(news_df, "news_daily_df")

# Key uniqueness
market_dupes = market_df.duplicated(subset=["session_date", "ticker"]).sum()
news_dupes = news_df.duplicated(subset=["session_date"]).sum()
record("market: unique on (session_date, ticker)", market_dupes == 0, f"{market_dupes} duplicate keys")
record("news: unique on session_date", news_dupes == 0, f"{news_dupes} duplicate keys")

# ticker <-> permno must be a stable 1:1 map (a break means a CRSP identifier changed mid-sample)
ticker_permno = market_df.groupby("ticker")["permno"].nunique()
record("market: one permno per ticker", (ticker_permno == 1).all(),
       f"{(ticker_permno > 1).sum()} tickers with multiple permnos")

# Nulls: forward-return columns are legitimately null at the tail of the sample (no future yet),
# so they are checked separately in section 4. Everything else must be complete.
core_market_cols = ["session_date", "ticker", "permno", "daily_return", "price", "volume", "asset"]
core_news_cols = [c for c in news_df.columns if c != "sentiment_bucket"]
record("market: no nulls in core columns", market_df[core_market_cols].isna().sum().sum() == 0,
       f"{int(market_df[core_market_cols].isna().sum().sum())} nulls")
record("news: no nulls in any column", news_df.isna().sum().sum() == 0,
       f"{int(news_df.isna().sum().sum())} nulls")

## 2. Quantity & coverage — is the panel balanced?

The eleven SPDR sector ETFs should each have a row on every trading session in the window, with one exception worth knowing about up front: **XLC and XLRE launched later than the others** (XLC in 2018, XLRE in 2015), so they should be *complete* across a 2020+ window — if either is short, that is a real gap, not a listing artifact.

The heatmap below is the fastest way to see a hole: any month/ticker cell that is lighter than its row neighbours is a month where that ETF is missing sessions.

In [ ]:
market_df["month"] = market_df["session_date"].values.astype("datetime64[M]")
coverage = market_df.pivot_table(index="ticker", columns="month", values="session_date", aggfunc="count")

# Sessions actually observed anywhere in the file = the empirical trading calendar
all_sessions = pd.DatetimeIndex(sorted(market_df["session_date"].unique()))
sessions_per_month = pd.Series(1, index=all_sessions).resample("MS").sum()
expected = coverage.copy()
for m in expected.columns:
    expected[m] = sessions_per_month.get(m, np.nan)
completeness = (coverage / expected)  # 1.0 = every session present for that ticker-month

fig, ax = plt.subplots(figsize=(13, 3.6))
im = ax.imshow(completeness.values, aspect="auto", cmap=SEQ, vmin=0, vmax=1)
ax.set_yticks(range(len(completeness.index)), completeness.index)
tick_pos = range(0, len(completeness.columns), 3)
ax.set_xticks(list(tick_pos), [completeness.columns[i].strftime("%Y-%m") for i in tick_pos], rotation=45, ha="right")
ax.set_title("Session coverage per sector ETF, by month (1.0 = every trading session present)")
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.02)
cbar.set_label("share of month's sessions present", color=MUTED)
# Mark any incomplete cell explicitly — colour alone should not carry a defect.
for y, x in zip(*np.where(completeness.values < 1.0)):
    ax.text(x, y, "x", ha="center", va="center", color=RED, fontsize=7, fontweight="bold")
plt.tight_layout()
plt.show()

rows_per_ticker = market_df["ticker"].value_counts().sort_index()
print(f"\nSessions in file: {len(all_sessions):,}   Tickers: {market_df['ticker'].nunique()}")
display(rows_per_ticker.to_frame("rows").T)

incomplete = int((completeness.fillna(0) < 1.0).sum().sum())
record("market: panel is balanced (all tickers on all sessions)",
       rows_per_ticker.nunique() == 1 and len(market_df) == len(all_sessions) * market_df["ticker"].nunique(),
       f"{incomplete} incomplete ticker-months; row counts per ticker: {sorted(rows_per_ticker.unique())}")
record("market: 11 sector ETFs present", market_df["ticker"].nunique() == 11,
       f"{market_df['ticker'].nunique()} tickers: {', '.join(sorted(market_df['ticker'].unique()))}")

## 3. Timestamp integrity

The highest-risk part of the pipeline. Three independent checks, from the raw event timestamp forward to the return label.

### 3a. Did the 4:00 PM ET cutoff actually get applied?

`01_` computed `signal_calendar_date` in SQL: an event stamped **before** 16:00 ET keeps its own ET date; an event stamped **at or after** 16:00 ET rolls to the next calendar day. Rather than trust the SQL, re-derive the rule in pandas from `timestamp_utc` and compare row by row. This reads the silver table's two timestamp columns only.

In [ ]:
HAVE_SILVER = CORE_EVENTS_CSV.exists()

if not HAVE_SILVER:
    print(f"Silver table not found at {CORE_EVENTS_CSV} — skipping 3a/3b.")
    print("Re-run 01_ravenpack_news_extraction.ipynb to regenerate it (it is gitignored by design).")
else:
    events_ts = pd.read_csv(
        CORE_EVENTS_CSV,
        usecols=["timestamp_utc", "signal_calendar_date"],  # timestamps only — no headline/event_text
        parse_dates=["timestamp_utc", "signal_calendar_date"],
    )

    # UTC -> America/New_York, honouring DST (the whole point of not hardcoding a -5h offset)
    ts_utc = events_ts["timestamp_utc"]
    ts_utc = ts_utc.dt.tz_localize("UTC") if ts_utc.dt.tz is None else ts_utc.dt.tz_convert("UTC")
    events_ts["ts_et"] = ts_utc.dt.tz_convert("America/New_York")
    events_ts["et_date"] = events_ts["ts_et"].dt.tz_localize(None).dt.normalize()
    events_ts["et_hour"] = events_ts["ts_et"].dt.hour + events_ts["ts_et"].dt.minute / 60
    events_ts["after_hours"] = events_ts["et_hour"] >= MARKET_CLOSE_HOUR_ET

    expected_cal_date = events_ts["et_date"] + pd.to_timedelta(events_ts["after_hours"].astype(int), unit="D")
    cutoff_mismatch = int((expected_cal_date != events_ts["signal_calendar_date"]).sum())

    print(f"Events in silver table: {len(events_ts):,}")
    print(f"After-hours events (>= 16:00 ET): {events_ts['after_hours'].sum():,} "
          f"({events_ts['after_hours'].mean():.1%})")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

    # Left: when do events actually land, in ET? The cutoff line should split the mass sensibly.
    ax = axes[0]
    bins = np.arange(0, 25, 0.5)
    intraday = events_ts.loc[~events_ts["after_hours"], "et_hour"]
    overnight = events_ts.loc[events_ts["after_hours"], "et_hour"]
    ax.hist(intraday, bins=bins, color=BLUE, label="same session (< 16:00 ET)")
    ax.hist(overnight, bins=bins, color=GRAY, label="rolled to next day (>= 16:00 ET)")
    ax.axvline(MARKET_CLOSE_HOUR_ET, color=RED, lw=2)
    ax.text(MARKET_CLOSE_HOUR_ET + 0.2, ax.get_ylim()[1] * 0.92, "16:00 ET close", color=RED, fontsize=8, fontweight="bold")
    ax.set_title("Event publish hour (ET) vs. the after-hours cutoff")
    ax.set_xlabel("hour of day, Eastern Time")
    ax.set_ylabel("events")
    ax.set_xticks(range(0, 25, 3))
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="y")
    ax.set_axisbelow(True)

    # Right: lag in calendar days from publish to assigned signal date. Must be 0 or 1, never negative.
    ax = axes[1]
    lag_days = (events_ts["signal_calendar_date"] - events_ts["et_date"]).dt.days
    lag_counts = lag_days.value_counts().sort_index()
    ax.bar(lag_counts.index, lag_counts.values, color=BLUE, width=0.6)
    for x, y in lag_counts.items():
        ax.text(x, y, f"{y:,}", ha="center", va="bottom", fontsize=8, color=MUTED)
    ax.set_title("Calendar-day lag: signal_calendar_date - publish date (ET)")
    ax.set_xlabel("days (0 = same day, 1 = rolled forward; negative = LOOKAHEAD BUG)")
    ax.set_ylabel("events")
    ax.set_xticks(sorted(lag_counts.index))
    ax.margins(y=0.15)
    ax.grid(axis="y")
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

    record("news: 16:00 ET cutoff correctly applied to every event", cutoff_mismatch == 0,
           f"{cutoff_mismatch:,} events whose signal_calendar_date disagrees with the recomputed rule")
    record("news: no event assigned to a date before it was published", (lag_days >= 0).all(),
           f"min lag = {int(lag_days.min())} day(s)")

### 3b. Does every event reach a trading session that is on or after its publish date?

`01_` mapped each `signal_calendar_date` forward to the next NYSE session (so weekend/holiday news rolls into Monday). Verified here against the **empirical CRSP trading calendar** from `market_daily_df` — i.e. the sessions we actually have returns for, which is the calendar that matters for the join.

The weekday panel is the visual tell: news published on a Saturday or Sunday must show up under a Monday session, never a same-day one.

In [ ]:
session_index = all_sessions  # CRSP sessions = ground-truth trading calendar

# Every news session_date must be a real trading session — with one legitimate exception.
# After-hours news on the FINAL day of the sample rolls forward to the next NYSE session,
# which CRSP has not published a price for yet. That trailing row is expected, and the left
# join in section 6 drops it. An orphan INSIDE the price window is a real calendar bug.
orphans = news_df.loc[~news_df["session_date"].isin(session_index), "session_date"]
last_market_session = session_index.max()
trailing_orphans = orphans[orphans > last_market_session]
in_window_orphans = orphans[orphans <= last_market_session]

record("news: every in-window session_date is a real CRSP trading session", len(in_window_orphans) == 0,
       f"{len(in_window_orphans)} non-session dates inside the price window" +
       (f", e.g. {in_window_orphans.head(3).dt.date.tolist()}" if len(in_window_orphans) else ""))
if len(trailing_orphans):
    print(f"note: {len(trailing_orphans)} trailing news session(s) past the last price session "
          f"({last_market_session.date()}): {trailing_orphans.dt.date.tolist()} — expected, dropped by the join.")

if HAVE_SILVER:
    # Re-derive the calendar -> next-session mapping and confirm each event's session is >= its publish date.
    cal_dates = pd.DatetimeIndex(sorted(events_ts["signal_calendar_date"].unique()))
    pos = session_index.searchsorted(cal_dates, side="left")  # first session on/after the calendar date
    mapped = pd.Series(
        [session_index[i] if i < len(session_index) else pd.NaT for i in pos],
        index=cal_dates, name="session_date",
    )
    events_ts["mapped_session"] = events_ts["signal_calendar_date"].map(mapped)

    unmapped = int(events_ts["mapped_session"].isna().sum())  # events past the end of the price sample
    mapped_ok = events_ts.dropna(subset=["mapped_session"])
    before_publish = int((mapped_ok["mapped_session"] < mapped_ok["et_date"]).sum())

    record("news: no event maps to a session before its ET publish date", before_publish == 0,
           f"{before_publish:,} lookahead events; {unmapped:,} events fall past the last price session")

    # Weekday flow: publish weekday (ET) -> assigned session weekday
    wd = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    flow = pd.crosstab(mapped_ok["et_date"].dt.dayofweek, mapped_ok["mapped_session"].dt.dayofweek)
    flow.index = [wd[i] for i in flow.index]
    flow.columns = [wd[i] for i in flow.columns]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

    ax = axes[0]
    im = ax.imshow(flow.values, cmap=SEQ, aspect="auto")
    ax.set_xticks(range(len(flow.columns)), flow.columns)
    ax.set_yticks(range(len(flow.index)), flow.index)
    ax.set_xlabel("assigned session weekday")
    ax.set_ylabel("publish weekday (ET)")
    ax.set_title("Where does news land? Publish weekday -> session weekday")
    ax.grid(False)
    for y in range(flow.shape[0]):
        for x in range(flow.shape[1]):
            v = flow.values[y, x]
            if v:
                ax.text(x, y, f"{v/1000:.0f}k" if v >= 1000 else f"{v}", ha="center", va="center", fontsize=7,
                        color="white" if v > flow.values.max() * 0.55 else INK)

    ax = axes[1]
    news_per_session = mapped_ok.groupby("mapped_session").size()
    ax.bar(news_per_session.index, news_per_session.values, color=BLUE, width=1.0)
    ax.set_title("Events per trading session (spikes = weekend/holiday roll-forward)")
    ax.set_ylabel("events assigned to session")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y")
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

### 3c. Are the forward returns actually forward?

The label columns are where a sign or shift error becomes an inflated accuracy score. Recompute `fwd_1d_return` and `fwd_5d_return` from `daily_return` independently — per ticker, sorted by session — and diff against the stored values. `fwd_5d` is checked against both the compounded-return definition and the simple price-ratio definition, so we learn which one `02_` used rather than assuming.

Also verified: the forward columns are null **only** at the tail of the sample (the last 1 and 5 sessions), which is the sole legitimate reason for them to be missing.

In [ ]:
chk = market_df.sort_values(["ticker", "session_date"]).copy()
g = chk.groupby("ticker", group_keys=False)

# fwd_1d: next session's daily return
chk["fwd_1d_recomputed"] = g["daily_return"].shift(-1)

# fwd_5d, definition A: compounded next 5 daily returns
gross = 1 + chk["daily_return"]
chk["_gross"] = gross
compounded = None
for k in range(1, 6):
    shifted = chk.groupby("ticker", group_keys=False)["_gross"].shift(-k)
    compounded = shifted if compounded is None else compounded * shifted
chk["fwd_5d_compounded"] = compounded - 1

# fwd_5d, definition B: price 5 sessions ahead / price today - 1
chk["fwd_5d_price_ratio"] = chk.groupby("ticker", group_keys=False)["price"].shift(-5) / chk["price"] - 1


def max_abs_diff(a, b):
    both = chk[[a, b]].dropna()
    return float((both[a] - both[b]).abs().max()) if len(both) else np.nan


d_1d = max_abs_diff("fwd_1d_return", "fwd_1d_recomputed")
d_5d_comp = max_abs_diff("fwd_5d_return", "fwd_5d_compounded")
d_5d_price = max_abs_diff("fwd_5d_return", "fwd_5d_price_ratio")

print(f"max |stored - recomputed| fwd_1d_return                : {d_1d:.3e}")
print(f"max |stored - recomputed| fwd_5d_return (compounded)   : {d_5d_comp:.3e}")
print(f"max |stored - recomputed| fwd_5d_return (price ratio)  : {d_5d_price:.3e}")

# Negative-lag falsification: if fwd_1d_return were accidentally the *previous* day's return,
# it would match a +1 shift instead. It must not.
chk["lagged_1d"] = g["daily_return"].shift(1)
d_lag = max_abs_diff("fwd_1d_return", "lagged_1d")
same_day = max_abs_diff("fwd_1d_return", "daily_return")
print(f"\nsanity: fwd_1d vs PREVIOUS day return (should be large): {d_lag:.3e}")
print(f"sanity: fwd_1d vs SAME day return (should be large)    : {same_day:.3e}")

record("market: fwd_1d_return == next session's daily_return", d_1d < TOL, f"max abs diff {d_1d:.2e}")
record("market: fwd_5d_return matches a forward 5-session definition",
       min(d_5d_comp, d_5d_price) < 1e-4,
       f"compounded {d_5d_comp:.2e} / price-ratio {d_5d_price:.2e}")
record("market: fwd_1d_return is not the same-day or lagged return",
       (d_lag > TOL) and (same_day > TOL), "forward shift confirmed, not a copy of a past/current return")

# Direction label consistency + tail-only nulls
label_ok = chk.dropna(subset=["fwd_1d_return", "fwd_1d_positive"])
mismatch = int(((label_ok["fwd_1d_return"] > 0).astype(float) != label_ok["fwd_1d_positive"]).sum())
record("market: fwd_1d_positive == (fwd_1d_return > 0)", mismatch == 0, f"{mismatch} mislabelled rows")

last_1 = chk["session_date"].drop_duplicates().nlargest(1)
last_5 = chk["session_date"].drop_duplicates().nlargest(5)
null_1d_off_tail = int(chk.loc[chk["fwd_1d_return"].isna() & ~chk["session_date"].isin(last_1)].shape[0])
null_5d_off_tail = int(chk.loc[chk["fwd_5d_return"].isna() & ~chk["session_date"].isin(last_5)].shape[0])
record("market: forward-return nulls occur only at the sample tail",
       null_1d_off_tail == 0 and null_5d_off_tail == 0,
       f"fwd_1d: {null_1d_off_tail} off-tail nulls; fwd_5d: {null_5d_off_tail} off-tail nulls")

## 4. Market data — do the numbers look like real ETF returns?

Daily sector-ETF returns should be roughly symmetric, centred near zero, mostly inside ±3%, with a fat left tail concentrated in **March 2020** (COVID crash). If the March 2020 cluster is missing, or if extremes show up at implausible dates, the return series is suspect. Prices must be positive and volume non-negative.

In [ ]:
tickers = sorted(market_df["ticker"].unique())

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8))

# Left: return dispersion per sector. One series, one colour — the ticker axis carries identity.
ax = axes[0]
data = [market_df.loc[market_df["ticker"] == t, "daily_return"].dropna() for t in tickers]
bp = ax.boxplot(data, tick_labels=tickers, showfliers=True, patch_artist=True, widths=0.6,
                flierprops=dict(marker=".", markersize=2, markerfacecolor=GRAY, markeredgecolor="none", alpha=0.5),
                medianprops=dict(color=INK, lw=1.2))
for box in bp["boxes"]:
    box.set(facecolor=BLUE, alpha=0.55, edgecolor=BLUE, linewidth=1)
ax.axhline(0, color=GRAY, lw=1)
ax.set_title("Daily return distribution by sector ETF")
ax.set_ylabel("daily return")
ax.grid(axis="y")
ax.set_axisbelow(True)

# Right: when do the extremes happen? Should cluster in Mar-2020.
ax = axes[1]
extreme = market_df[market_df["daily_return"].abs() > 0.05]
ax.scatter(extreme["session_date"], extreme["daily_return"], s=10, alpha=0.6,
           color=[BLUE if r > 0 else RED for r in extreme["daily_return"]])
ax.axhline(0, color=GRAY, lw=1)
ax.set_title("Moves larger than +/-5% (blue = up, red = down)")
ax.set_ylabel("daily return")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="y")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

print("Ten largest absolute daily moves:")
display(
    market_df.reindex(market_df["daily_return"].abs().sort_values(ascending=False).index)
    .head(10)[["session_date", "ticker", "daily_return", "price", "volume"]]
    .assign(session_date=lambda d: d["session_date"].dt.date)
    .reset_index(drop=True)
)

bad_price = int((market_df["price"] <= 0).sum())
bad_volume = int((market_df["volume"] < 0).sum())
zero_volume = int((market_df["volume"] == 0).sum())
implausible = int((market_df["daily_return"].abs() > 0.25).sum())
mar2020 = extreme[extreme["session_date"].dt.to_period("M") == "2020-03"].shape[0]

record("market: all prices positive", bad_price == 0, f"{bad_price} rows with price <= 0")
record("market: no negative volume", bad_volume == 0, f"{bad_volume} rows negative, {zero_volume} rows zero")
record("market: no implausible daily returns (|r| > 25%)", implausible == 0, f"{implausible} rows")
record("market: COVID crash present in the extremes", mar2020 > 0,
       f"{mar2020} of {len(extreme)} >5% moves fall in March 2020")

## 5. News data — is the sentiment aggregate internally consistent and stable?

Three things to see: the sentiment shares sum to exactly 1 and the counts reconcile to `event_record_count`; **news volume per session is stable enough that later sessions are not thin** (a collapsing event count late in the sample usually means the WRDS table for that year is partially loaded); and the daily mean sentiment behaves like a sentiment series — noisy day to day, with a visible negative excursion around March 2020.

In [ ]:
share_cols = ["positive_event_share", "negative_event_share", "neutral_event_share"]
count_cols = ["positive_event_count", "negative_event_count", "neutral_event_count"]

share_sum = news_df[share_cols].sum(axis=1)
count_sum = news_df[count_cols].sum(axis=1)

record("news: sentiment shares sum to 1.0", np.allclose(share_sum, 1.0, atol=1e-6),
       f"max deviation {float((share_sum - 1).abs().max()):.2e}")
record("news: pos+neg+neutral counts == event_record_count",
       bool((count_sum == news_df["event_record_count"]).all()),
       f"{int((count_sum != news_df['event_record_count']).sum())} mismatched sessions")
record("news: unique_story_count <= event_record_count",
       bool((news_df["unique_story_count"] <= news_df["event_record_count"]).all()),
       "no session has more unique stories than records")
record("news: mean sentiment within [-1, 1]",
       bool(news_df["mean_event_sentiment_score"].between(-1, 1).all()),
       f"range [{news_df['mean_event_sentiment_score'].min():.3f}, {news_df['mean_event_sentiment_score'].max():.3f}]")

fig, axes = plt.subplots(3, 1, figsize=(12.5, 7.5), sharex=True)

# Volume of news per session, with a 21-session mean to expose thinning.
ax = axes[0]
ax.plot(news_df["session_date"], news_df["event_record_count"], color=GRAY, lw=0.7, alpha=0.7)
roll = news_df.set_index("session_date")["event_record_count"].rolling(21).mean()
ax.plot(roll.index, roll.values, color=BLUE, lw=2)
ax.text(roll.index[-1], roll.iloc[-1], "  21-session mean", color=BLUE, fontsize=8, va="center", fontweight="bold")
ax.set_title("News volume per trading session — watch for thinning at either end")
ax.set_ylabel("events")
ax.grid(axis="y")
ax.set_axisbelow(True)

# Mean sentiment per session.
ax = axes[1]
ax.plot(news_df["session_date"], news_df["mean_event_sentiment_score"], color=GRAY, lw=0.7, alpha=0.7)
roll_s = news_df.set_index("session_date")["mean_event_sentiment_score"].rolling(21).mean()
ax.plot(roll_s.index, roll_s.values, color=BLUE, lw=2)
ax.axhline(0, color=RED, lw=1, ls="--")
ax.text(roll_s.index[-1], roll_s.iloc[-1], "  21-session mean", color=BLUE, fontsize=8, va="center", fontweight="bold")
ax.set_title("Mean RavenPack event sentiment per session")
ax.set_ylabel("mean score")
ax.grid(axis="y")
ax.set_axisbelow(True)

# Composition: does the pos/neg mix drift? Stacked shares, diverging blue/gray/red.
ax = axes[2]
m = news_df.set_index("session_date")[share_cols].rolling(21).mean().dropna()
ax.stackplot(m.index, m["positive_event_share"], m["neutral_event_share"], m["negative_event_share"],
             colors=[BLUE, "#d8d7d1", RED], labels=["positive", "neutral", "negative"], alpha=0.9)
ax.set_ylim(0, 1)
ax.set_title("Sentiment mix per session (21-session mean of shares)")
ax.set_ylabel("share of events")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(False)

plt.tight_layout()
plt.show()

print("Events per session, by year:")
display(
    news_df.assign(year=news_df["session_date"].dt.year)
    .groupby("year")
    .agg(sessions=("session_date", "size"),
         events=("event_record_count", "sum"),
         events_per_session=("event_record_count", "mean"),
         sources_per_session=("unique_source_count", "mean"),
         mean_sentiment=("mean_event_sentiment_score", "mean"))
    .round(2)
)
print("sentiment_bucket balance:")
display(news_df["sentiment_bucket"].value_counts().to_frame("sessions").T)

# A session with a handful of events is too thin to score; flag rather than fail.
thin = int((news_df["event_record_count"] < 10).sum())
record("news: no critically thin sessions (< 10 events)", thin == 0,
       f"{thin} sessions below 10 events (min = {int(news_df['event_record_count'].min())})")

## 6. The join — does the news table actually cover the market panel?

The modelling table is `market_daily_df` left-joined to `news_daily_df` on `session_date`. What matters is how many market sessions get news, and whether the missing ones cluster (a systematic gap) or scatter (harmless).

The right-hand panel is the **model-free sanity check** from the project plan: average forward 1-day return stratified by the session's sentiment bucket. This is a smoke test, not a result, and the check is on **magnitude, not sign** — a bucket mean far outside the ±50 bps range would point at lookahead contamination rather than signal, because no daily sentiment split should separate next-day returns that cleanly.

Do not read the *direction* here as a preview of the finding. Negative-sentiment sessions plausibly earn a *higher* forward return (a next-day rebound after bad news) — that is a well-known contrarian pattern, and disentangling it from a genuine sentiment signal is exactly what the baseline-vs-augmented comparison in the modelling notebook is for. Note also that the buckets are heavily unbalanced (far more positive than negative sessions), so bucket means built on few sessions are noisy.

In [ ]:
panel = market_df.merge(news_df, on="session_date", how="left", validate="many_to_one")

market_sessions = set(all_sessions)
news_sessions = set(news_df["session_date"])
sessions_with_news = sorted(market_sessions & news_sessions)
sessions_without_news = sorted(market_sessions - news_sessions)
news_without_market = sorted(news_sessions - market_sessions)

coverage_pct = len(sessions_with_news) / len(market_sessions) * 100
print(f"Market sessions:            {len(market_sessions):,}")
print(f"  with news:                {len(sessions_with_news):,} ({coverage_pct:.1f}%)")
print(f"  without news:             {len(sessions_without_news):,}")
print(f"News sessions with no market data: {len(news_without_market):,} (trailing roll-forward past the price sample)")
if sessions_without_news:
    print(f"  first few uncovered sessions: {[d.date() for d in sessions_without_news[:8]]}")
print(f"\nJoined panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.6))

# Left: news coverage per month across the market calendar.
ax = axes[0]
cov = pd.DataFrame({"session_date": sorted(market_sessions)})
cov["has_news"] = cov["session_date"].isin(news_sessions)
monthly = cov.set_index("session_date")["has_news"].resample("MS").mean()
ax.bar(monthly.index, monthly.values * 100, width=22, color=BLUE)
ax.axhline(100, color=GRAY, lw=1, ls="--")
ax.set_ylim(0, 108)
ax.set_title("Share of trading sessions with news, by month")
ax.set_ylabel("% of sessions covered")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="y")
ax.set_axisbelow(True)

# Right: mean forward return by sentiment bucket (bps), one bar per bucket.
ax = axes[1]
order = ["negative", "neutral", "positive"]
strat = (panel.dropna(subset=["sentiment_bucket", "fwd_1d_return"])
         .groupby("sentiment_bucket")["fwd_1d_return"]
         .agg(["mean", "count"])
         .reindex(order).dropna())
bps = strat["mean"] * 10_000
colors = {"negative": RED, "neutral": GRAY, "positive": BLUE}
ax.bar(strat.index, bps.values, color=[colors[i] for i in strat.index], width=0.55)
for i, (label, v) in enumerate(bps.items()):
    ax.text(i, v, f"{v:+.1f} bps\nn={int(strat.loc[label, 'count']):,}",
            ha="center", va="bottom" if v >= 0 else "top", fontsize=8, color=MUTED)
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set_title("Mean forward 1-day return by session sentiment bucket")
ax.set_ylabel("mean fwd 1d return (bps)")
ax.margins(y=0.25)
ax.grid(axis="y")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

up_rate = float(panel["fwd_1d_positive"].dropna().mean())
print(f"Base rate — share of ticker-sessions with a positive next-day return: {up_rate:.1%}")
print("(This is what a baseline classifier that always predicts 'up' would score. Any model must beat it.)")

record("join: >= 95% of trading sessions have news", coverage_pct >= 95, f"{coverage_pct:.1f}% covered")
record("join: no news sessions orphaned inside the market calendar", len(in_window_orphans) == 0,
       f"{len(in_window_orphans)} in-window orphans; {len(trailing_orphans)} trailing (expected, dropped by the join)")
record("join: market:news is many-to-one (no row fan-out)", len(panel) == len(market_df),
       f"{len(panel):,} joined rows vs {len(market_df):,} market rows")
record("join: class balance is not degenerate", 0.40 <= up_rate <= 0.60,
       f"positive-class base rate {up_rate:.1%}")
record("join: sentiment/return gradient is plausible, not suspiciously large",
       bool(bps.abs().max() < 50), f"largest bucket mean {bps.abs().max():.1f} bps")

## 7. QA scorecard

Every check above, in one table. **Any `FAIL` is a bug in `01_`/`02_` and must be fixed there** — not patched downstream in the modelling notebook.

In [ ]:
scorecard = pd.DataFrame(CHECKS)
n_fail = int((scorecard["result"] == "FAIL").sum())

display(
    scorecard.style
    .map(lambda v: f"color: {'#d03b3b' if v == 'FAIL' else '#0ca30c'}; font-weight: 700", subset=["result"])
    .hide(axis="index")
)

print(f"\n{len(scorecard) - n_fail} passed / {n_fail} failed of {len(scorecard)} checks.")
if n_fail:
    print("\nFailed checks:")
    for _, row in scorecard[scorecard["result"] == "FAIL"].iterrows():
        print(f"  - {row['check']}: {row['detail']}")
    raise AssertionError(f"{n_fail} data-quality check(s) failed — see above before using this data.")
print("\nBoth gold tables are clean, complete, and correctly time-aligned. Cleared for feature engineering.")

## 8. Export the combined modelling table: `model_daily_panel.csv`

The single gold input for the modelling stage: one row per `(session_date, ticker)` — market features and forward-return targets from `market_daily_df`, joined to that session's news aggregates from `news_daily_df`. This cell sits *after* the scorecard on purpose: the scorecard cell raises on any failure, so this file can only ever be produced by a run in which **all** QA checks passed.

Why this join is leakage-free by construction: the news columns for session *t* summarize only events published **before 16:00 ET on day *t*** (the after-hours cutoff verified in section 3a), and the target `fwd_1d_return` is the return from the close of day *t* to the close of day *t+1*. The predictor set is therefore fully known at the moment the target window opens. What this table does **not** do — deliberately left to the modelling notebook: temporal train/validation/test partitioning (chronological blocks, never shuffled), feature lagging/rolling windows, and any LLM-derived sentiment columns (those join in later on `session_date`, or on `rp_story_id` at the event level).

Licensing: every column here is either exchange price data or an aggregated daily summary — no row-level RavenPack records — so the file is safe to commit.

In [ ]:
GOLD_PANEL_CSV = NOTEBOOK_DIR / "model_daily_panel.csv"

news_feature_cols = [c for c in news_df.columns if c != "session_date"]
gold_panel = (
    market_df.drop(columns=["month"])
    .merge(news_df, on="session_date", how="left", validate="many_to_one")
    .sort_values(["session_date", "ticker"])
    .reset_index(drop=True)
)

assert len(gold_panel) == len(market_df), "join fanned out or dropped market rows"
missing_news_rows = int(gold_panel[news_feature_cols].isna().any(axis=1).sum())
if missing_news_rows:
    print(
        f"Warning: {missing_news_rows:,} market rows have no news features; "
        "the modeling notebooks must exclude them."
    )

gold_panel.to_csv(GOLD_PANEL_CSV, index=False)

print(f"Wrote {GOLD_PANEL_CSV.name}: {gold_panel.shape[0]:,} rows x {gold_panel.shape[1]} cols")
print(f"  sessions: {gold_panel['session_date'].nunique():,}  "
      f"({gold_panel['session_date'].min().date()} to {gold_panel['session_date'].max().date()})")
print(f"  tickers:  {gold_panel['ticker'].nunique()}")
print(f"  rows with complete news features: {len(gold_panel) - missing_news_rows:,}")
print(f"  targets:  fwd_1d_return / fwd_1d_positive / fwd_5d_return / fwd_5d_positive "
      f"(null only at the sample tail: {int(gold_panel['fwd_1d_return'].isna().sum())} / "
      f"{int(gold_panel['fwd_5d_return'].isna().sum())} rows)")
print("\nColumns:", ", ".join(gold_panel.columns))